# Exploratory Data Analysis: Philippine Major Crops Production

This notebook performs Exploratory Data Analysis (EDA) on the Philippine Major Crops Production dataset.

## 1. Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

file_path = r'c:\Users\James\Music\CVPP\data\original\Major Crops- Volume of Production, by Region, Quarter, and Semester, 2010-2026.csv'
df = pd.read_csv(file_path, skiprows=1)

print(f"Dataset Shape: {df.shape}")
display(df.head())
df.info()

## 2. Data Quality Checks

Checking for missing values, zero-value entries, and duplicates.

In [ ]:
df.replace('..', np.nan, inplace=True)

missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts[missing_counts > 0])

duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

## 3. Reshaping

Converting from wide to long format (Crop, Region, Year, Quarter -> Volume).

In [ ]:
df_filtered = df[df['Geolocation'] != 'PHILIPPINES'].copy()
quarter_cols = [col for col in df_filtered.columns if 'Quarter' in col]
id_vars = ['Crop', 'Geolocation']

df_quarters = df_filtered[id_vars + quarter_cols]
df_long = pd.melt(df_quarters, id_vars=id_vars, value_vars=quarter_cols, var_name='Period', value_name='Volume')

df_long[['Year', 'Quarter']] = df_long['Period'].str.extract(r'(\d{4})\s*(Quarter\d)')
df_long['Year'] = df_long['Year'].astype(int)
df_long.drop(columns=['Period'], inplace=True)

if df_long['Volume'].dtype == 'object':
    df_long['Volume'] = df_long['Volume'].astype(str).str.replace(',', '')
df_long['Volume'] = pd.to_numeric(df_long['Volume'], errors='coerce')

display(df_long.head())

## 4. Descriptive Statistics

In [ ]:
display(df_long['Volume'].describe())

zero_volume = (df_long['Volume'] == 0).sum()
print(f"Number of zero-volume entries: {zero_volume}")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df_long['Volume'].dropna() + 1, bins=50, log_scale=True, ax=axes[0])
axes[0].set_title('Distribution of Production Volume (Log Scale)')
axes[0].set_xlabel('Volume (Log)')

sns.boxplot(x=np.log1p(df_long['Volume'].dropna()), ax=axes[1])
axes[1].set_title('Boxplot of Production Volume (Log Scale)')
axes[1].set_xlabel('Volume (Log)')

plt.tight_layout()
plt.show()

In [ ]:
crop_totals = df_long.groupby('Crop')['Volume'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=crop_totals.values, y=crop_totals.index, palette='viridis')
plt.title('Top 10 Crops by Total Production Volume')
plt.xlabel('Total Volume')
plt.ylabel('Crop')
plt.show()

In [ ]:
region_totals = df_long.groupby('Geolocation')['Volume'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=region_totals.values, y=region_totals.index, palette='magma')
plt.title('Top 10 Regions by Total Production Volume')
plt.xlabel('Total Volume')
plt.ylabel('Region')
plt.show()

In [ ]:
yearly_trend = df_long.groupby('Year')['Volume'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=yearly_trend, x='Year', y='Volume', marker='o')
plt.title('Total Production Trend Over Time')
plt.xlabel('Year')
plt.ylabel('Total Volume')
plt.xticks(yearly_trend['Year'].unique(), rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_long, x='Quarter', y=np.log1p(df_long['Volume']), palette='Set2')
plt.title('Quarterly Seasonal Pattern (Log Scale)')
plt.ylabel('Volume (Log)')
plt.show()

In [ ]:
top_crops = crop_totals.index[:15]
heatmap_data = df_long[df_long['Crop'].isin(top_crops)].groupby(['Geolocation', 'Crop'])['Volume'].sum().unstack().fillna(0)

plt.figure(figsize=(14, 10))
sns.heatmap(np.log1p(heatmap_data), cmap='YlGnBu', annot=False)
plt.title('Log of Total Production: Region vs Top 15 Crops')
plt.xlabel('Crop')
plt.ylabel('Region')
plt.show()

In [ ]:
correlation_matrix = df_long[['Year', 'Volume']].corr()
display(correlation_matrix)

plt.figure(figsize=(5, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

## 6. Key Findings & Insights

1. **Data Quality**: `..` entries representing missing values were replaced with NaN. Some records might have zero volume, requiring handling depending on the model.
2. **Distribution**: Volume distribution is extremely right-skewed. Transforming targets (e.g., log1p) is advised for modeling.
3. **Top Features**: A few crops (e.g., Sugarcane, Palay, Corn) and regions dominate production totals.
4. **Seasonality**: There's noticeable variation in production across quarters, reflecting planting/harvesting cycles.
5. **Modeling approach**: Tree-based models (XGBoost, LightGBM) would be suitable as they are robust to skewed targets and complex interactions between Crop, Region, and Season. Alternatively, panel time series models could be used to predict future quarters.